In [ ]:
def save_frames_as_images(ds, save_folder, img_size, meta_path):
    # 동아시아 영역 자르기 (위도 방향 주의)
    ds_eastasia = ds.sel(lat=slice(33, 43), lon=slice(124, 132))

    # lev 차원 제거 (있으면)
    if 'lev' in ds_eastasia['precipitation'].dims:
        data = ds_eastasia['precipitation'].squeeze('lev').values
    else:
        data = ds_eastasia['precipitation'].values
    
    # 결측치 -> 0
    data = np.nan_to_num(data, nan=0.0)

    # 최소/최대값
    min_val = float(data.min())
    max_val = float(data.max())

    # 0~255 정규화
    normed_data = (data - min_val) / (max_val - min_val) * 255
    normed_data = normed_data.astype(np.uint8)

    # 폴더 생성
    os.makedirs(save_folder, exist_ok=True)

    # 프레임 이미지 저장
    for idx, frame in enumerate(tqdm(normed_data)):
        img = Image.fromarray(np.flipud(frame))  # 위아래 반전
        img = img.resize(img_size, resample=Image.BILINEAR)
        img = img.convert('RGB')
        img.save(os.path.join(save_folder, f"{idx:05d}.png"))

    # 메타데이터 저장
    metadata = {
        "min_val": min_val,
        "max_val": max_val,
        'uint': "mm/day",
        "lat_range": [float(ds_eastasia.lat.max()), float(ds_eastasia.lat.min())],
        "lon_range": [float(ds_eastasia.lon.min()), float(ds_eastasia.lon.max())],
        'original_shape': list(data.shape),
        'time': [str(t) for t in ds_eastasia.time.values]
    }

    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=4)


In [ ]:
# imerg
import os, json
import numpy as np
from tqdm import tqdm
from PIL import Image

def save_frames_as_images(ds, save_folder, img_size, meta_path):
    # 사용할 변수 선택
    da = ds['precipitation']      # DataArray

    # 동아시아 영역 자르기
    da = da.sel(lat=slice(33, 43), lon=slice(124, 134))

    # lev 차원 제거 (있으면)
    if 'lev' in da.dims:
        da = da.squeeze('lev')

    # 차원 순서를 (time, lat, lon)으로 통일
    # IMERG는 보통 (time, lon, lat)이니까 여기서 순서 맞춰줌
    want_dims = [d for d in ['time', 'lat', 'lon'] if d in da.dims]
    da = da.transpose(*want_dims)

    # 위도를 북→남(큰 위도→작은 위도) 순서로 정렬
    # 그림 그릴 때 위쪽이 북쪽이 되게 하고 싶으면 켜기
    if da['lat'][0] < da['lat'][-1]:
        da = da.sortby('lat', ascending=False)

    # numpy 배열로 변환: (time, lat, lon)
    data = da.values

    # 결측치 → 0
    data = np.nan_to_num(data, nan=0.0)

    # 최소/최대값
    min_val = float(data.min())
    max_val = float(data.max())

    # 0~255로 정규화
    normed_data = (data - min_val) / (max_val - min_val + 1e-12) * 255
    normed_data = normed_data.astype(np.uint8)

    # 폴더 생성
    os.makedirs(save_folder, exist_ok=True)
    os.makedirs(os.path.dirname(meta_path), exist_ok=True)

    # 프레임 이미지 저장
    for idx, frame in enumerate(tqdm(normed_data)):
        # 이제 frame.shape == (lat, lon) == (H, W)  ✅
        img = Image.fromarray(frame)
        img = img.resize(img_size, resample=Image.BILINEAR)
        img = img.convert('L')
        img.save(os.path.join(save_folder, f"{idx:05d}.png"))

    # 메타데이터 저장
    metadata = {
        "min_val": min_val,
        "max_val": max_val,
        "unit": "mm/day",
        "lat_range": [float(da['lat'].max()), float(da['lat'].min())],
        "lon_range": [float(da['lon'].min()), float(da['lon'].max())],
        "original_shape": list(data.shape),  # (time, lat, lon)
        "time": [str(t) for t in da['time'].values]
    }

    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=4)


In [ ]:
import os
# 경로설정
LR_PATH = "../Siren_pt/data/ERA.mtpr.195001_201912.nc"
HR_PATH = "../Siren_pt/data/mtpr_195001_201912_1deg.nc"
SAVE_DIR = "./load/climate"
os.makedirs(os.path.join(SAVE_DIR, 'climate_train_LR'), exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, 'climate_train_HR'), exist_ok=True)
# 메타데이터 폴더 생성
# os.makedirs(os.path.dirname(meta_path), exist_ok=True)


NameError: name 'os' is not defined

In [5]:
# ds_lr = xr.open_dataset(LR_PATH)
# ds_hr = xr.open_dataset(HR_PATH)
import os
import json
import numpy as np
import xarray as xr
from tqdm import tqdm
from PIL import Image

path = "../../../data/IMERG/Imerg_raw/imerg_merged_precip.nc"
ds = xr.open_dataset(path)

SAVE_DIR = "./load/climate"
# os.makedirs(os.path.join(SAVE_DIR, 'climate_train_korea_imerg50'), exist_ok=True)

# === Train ===
train_time_slice = slice('1998-01-01', '2009-12-31')
save_frames_as_images(
    ds.sel(time=train_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_train_korea_imerg50_test'),
    img_size=(50, 50),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'train_imerg50_meta_test.json')
)

save_frames_as_images(
    ds.sel(time=train_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_train_korea_imerg100_test'),
    img_size=(100, 100),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'train_imerg100_meta_test.json')
)

# save_frames_as_images(
#     ds.sel(time=train_time_slice),
#     save_folder=os.path.join(SAVE_DIR, 'climate_train_korea_imerg100_'),
#     img_size=(100, 100),
#     meta_path=os.path.join(SAVE_DIR, 'meta', 'train_imerg200_meta_132.json')
# )

# # === Validation ===
val_time_slice = slice('2010-01-01', '2019-12-31')
save_frames_as_images(
    ds.sel(time=val_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_val_korea_imerg50_test'),
    img_size=(50, 50),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'val_imerg50_meta_test.json')
)
save_frames_as_images(
    ds.sel(time=val_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_val_korea_imerg100_test'),
    img_size=(100, 100),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'val_imerg100_meta_test.json')
)

# save_frames_as_images(
#     ds.sel(time=val_time_slice),
#     save_folder=os.path.join(SAVE_DIR, 'climate_val_korea_imerg200_132'),
#     img_size=(160, 200),
#     meta_path=os.path.join(SAVE_DIR, 'meta', 'val_imerg200_meta_132.json')
# )


100%|██████████| 3652/3652 [00:01<00:00, 2413.11it/s]


In [2]:
import xarray as xr
import os
import pandas as pd

folder_reanalysis = "../data/Imerg_raw/imerg_merged_precip.nc"
folder_obs = "data/preprocessed_daily_precip_data.csv"

ds = xr.open_dataset(folder_reanalysis)
df = ds1.to_dataframe()
display(df['precip'].max())

ds2 = pd.read_csv(folder_obs)
display(ds2.v.max())

# ds = ds1.sel(latitude=slice(20, 50), longitude=slice(100, 150))
# # ds = ds1.sel(time=slice('1996-01-01', '2010-12-31'))
# display(ds.precip.min().item(), ds['precip'].max().item())
# display(ds2['v'].min(),ds2['v'].max())

FileNotFoundError: [Errno 2] No such file or directory: '/home/inhye_yoo/deeplearning/inr/data/Imerg_raw/imerg_merged_precip.nc'

In [ ]:
# 1. xarray → pandas 변환
df = ds['precipitation']
df

# # 2. NaN 제거
# df = df.dropna(subset=['precip'])

# # 3. datetime 변환 (안 돼 있으면)
# df['time'] = pd.to_datetime(df['time'], errors='coerce')

# # 4. 원하는 기간 필터링
# df_filtered = df[(df['time'] >= '1996-10-01') & (df['time'] <= '2009-12-31')]

# # 5. 날짜별 관측 개수 출력
# print(df_filtered.groupby('time').size())

<xarray.DataArray 'precip' (time: 10319, latitude: 31, longitude: 51)> Size: 65MB
array([[[0.      , 0.      , ..., 0.677041, 0.363582],
        [0.      , 0.      , ..., 0.471274, 0.443039],
        ...,
        [0.147724, 0.311034, ..., 0.      , 0.078113],
        [0.636648, 0.214771, ..., 0.      , 0.      ]],

       [[0.      , 0.      , ..., 0.      , 0.      ],
        [0.      , 0.      , ..., 0.      , 0.      ],
        ...,
        [0.451633, 0.268609, ..., 0.      , 0.343169],
        [0.729142, 1.121461, ..., 0.      , 0.      ]],

       ...,

       [[0.      , 0.      , ..., 0.      , 0.      ],
        [0.      , 0.      , ..., 0.076106, 0.011766],
        ...,
        [0.      , 0.      , ..., 0.      , 0.      ],
        [0.      , 0.      , ..., 0.      , 0.      ]],

       [[0.      , 0.      , ..., 0.      , 0.      ],
        [0.      , 0.      , ..., 0.      , 0.      ],
        ...,
        [0.      , 0.      , ..., 0.      , 0.      ],
        [0.      , 0.      , ..., 0.      , 0.      ]]], dtype=float32)
Coordinates:
  * time       (time) datetime64[ns] 83kB 1996-10-01 1996-10-02 ... 2024-12-31
  * longitude  (longitude) float32 204B 100.0 101.0 102.0 ... 148.0 149.0 150.0
  * latitude   (latitude) float32 124B 20.0 21.0 22.0 23.0 ... 48.0 49.0 50.0
Attributes:
    standard_name:  lwe_precipitation_rate
    long_name:      NOAA Climate Data Record (CDR) of Daily GPCP Satellite-Ga...
    units:          mm/day
    cell_methods:   area: mean time: mean

In [5]:
SAVE_DIR = "load/climate"
# daily_precip_path = "C:/Temp/precip.day.mean.nc"
# os.makedirs(os.path.join(SAVE_DIR, 'climate_daily_1deg'))


In [6]:
import os
import json
import numpy as np
import xarray as xr
from tqdm import tqdm
from PIL import Image



ds = xr.open_dataset(folder_reanalysis)

train_time_slice = slice('2024-06-01', '2024-08-31')
save_frames_as_images(
    ds.sel(time = train_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_train_daily30_06_08'),
    img_size=(30, 30),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'daily_train_30_06_08.json')
)

val_time_slice = slice('2024-09-01', '2024-12-31')
save_frames_as_images(
    ds.sel(time = val_time_slice),
    save_folder=os.path.join(SAVE_DIR, 'climate_val_daily30_06_08'),
    img_size=(30, 30),
    meta_path=os.path.join(SAVE_DIR, 'meta', 'daily_val_60_06_08.json')
)

  0%|          | 0/92 [00:00<?, ?it/s]

100%|██████████| 122/122 [00:00<00:00, 3545.21it/s]


## obs 데이터 전처리

In [ ]:
import os
from tqdm import tqdm
from PIL import Image
import numpy as np
import pandas as pd

# ===== 설정 =====
CSV_PATH = "data/preproccessed_obs_2010_2019.csv"
SAVE_DIR = "load/climate/obs_1deg_60x60_2010"
H, W = 60, 60  # 60개의 정수 위도(20..79), 60개의 정수 경도(100..159)
LAT_MIN, LAT_MAX = 20, 79     # 위도 정수 그리드
LON_MIN, LON_MAX = 100, 159   # 경도 정수 그리드
NORM_MODE = "global"          # 'global' 또는 'per_time'
FLIP_UD = True               # 최종 이미지 위아래 반전(이전 파이프라인과 맞추려면 True/False 선택)

os.makedirs(SAVE_DIR, exist_ok=True)

def to_index_1deg(lat, lon):
    """
    관측 (lat, lon)을 가장 가까운 정수 위·경도로 스냅 → 60x60 인덱스(r,c) 반환.
    r: 0이 북쪽(=79), 59가 남쪽(=20)
    c: 0이 서쪽(=100), 59가 동쪽(=159)
    """
    lat_i = np.rint(lat).astype(int)
    lon_i = np.rint(lon).astype(int)
    lat_i = np.clip(lat_i, LAT_MIN, LAT_MAX)
    lon_i = np.clip(lon_i, LON_MIN, LON_MAX)

    r = (LAT_MAX - lat_i).astype(int)      # 79→0, 20→59
    c = (lon_i - LON_MIN).astype(int)      # 100→0, 159→59
    r = np.clip(r, 0, H-1); c = np.clip(c, 0, W-1)
    return r, c

# ===== CSV 로드 & 정리 =====
df = pd.read_csv(CSV_PATH)
for junk in ['Unnamed: 0', 'index']:
    if junk in df.columns: df = df.drop(columns=junk)

# 여기서는 x=위도(lat), y=경도(lon)으로 가정
needed = ['time', 'x', 'y', 'v']
for col in needed:
    if col not in df.columns:
        raise ValueError(f"CSV에 '{col}' 컬럼이 필요합니다.")

df['time'] = pd.to_datetime(df['time'])
df['x'] = pd.to_numeric(df['x'], errors='coerce')   # lat
df['y'] = pd.to_numeric(df['y'], errors='coerce')   # lon
df['v'] = pd.to_numeric(df['v'], errors='coerce').fillna(0.0)
df = df.dropna(subset=['x','y'])

# 정규화 기준
if NORM_MODE == 'global':
    global_min = float(df['v'].min())
    global_max = float(df['v'].max())
    global_den = (global_max - global_min) if global_max > global_min else 1.0

# ===== 날짜별 60×60 생성 =====
times = df['time'].sort_values().unique()
for t in tqdm(times, desc="saving 1deg 60x60"):
    g = df[df['time'] == t]

    # 60×60 그리드 누적(같은 칸 여러 점 → 평균)
    grid_sum   = np.zeros((H, W), dtype=np.float32)
    grid_count = np.zeros((H, W), dtype=np.int32)

    if not g.empty:
        # 범위 밖 정수 스냅까지 고려해 필터(선택)
        g = g[(g['x'] >= LAT_MIN-0.5) & (g['x'] <= LAT_MAX+0.5) &
              (g['y'] >= LON_MIN-0.5) & (g['y'] <= LON_MAX+0.5)]

        if not g.empty:
            lat = g['x'].to_numpy(np.float64)
            lon = g['y'].to_numpy(np.float64)
            val = g['v'].to_numpy(np.float32, copy=False)

            r, c = to_index_1deg(lat, lon)
            np.add.at(grid_sum,   (r, c), val)
            np.add.at(grid_count, (r, c), 1)

    grid = grid_sum / np.maximum(1, grid_count)
    grid[grid_count == 0] = 0.0

    # ---- 정규화 → 0..255 (uint8) ----
    if NORM_MODE == 'global':
        grid_01 = (grid - global_min) / global_den
    else:
        gmin, gmax = float(np.nanmin(grid)), float(np.nanmax(grid))
        den = (gmax - gmin) if gmax > gmin else 1.0
        grid_01 = (grid - gmin) / den
    grid_01 = np.clip(grid_01, 0.0, 1.0)
    grid_u8 = (grid_01 * 255.0).astype(np.uint8)

    if FLIP_UD:
        grid_u8 = np.flipud(grid_u8)  # 필요 시 북쪽=위로 맞추기

    img = Image.fromarray(grid_u8).convert('RGB')   # (60,60) → RGB
    fname = pd.to_datetime(t).strftime("%Y%m%d") + ".png"
    img.save(os.path.join(SAVE_DIR, fname))


saving 1deg 60x60: 100%|██████████| 3652/3652 [00:05<00:00, 610.80it/s]


In [7]:
import pandas as pd

# 1. 고정된 지점 목록 (NaN 없이)
df['time'] = pd.to_datetime(df['time'], errors='coerce')
site_coords = df[['x', 'y']].drop_duplicates().reset_index(drop=True)

# 2. 전체 날짜 목록
all_times = pd.date_range(start='1996-10-01', end='2009-12-31', freq='D')

# 3. 모든 (time, x, y) 조합 생성
full_grid = pd.DataFrame([
    {'time': time, 'x': x, 'y': y}
    for time in all_times
    for x, y in site_coords.to_numpy()
])

# 4. 기존 데이터와 left join
df_filled = full_grid.merge(df, on=['time', 'x', 'y'], how='left')

# 결과 확인
print(df_filled)

             time        x         y  Unnamed: 0    v
0      1996-10-01  38.2509  128.5647           0  NaN
1      1996-10-01  38.1479  127.3042           1  NaN
2      1996-10-01  37.9019  127.0607           2  NaN
3      1996-10-01  37.8859  126.7665           3  NaN
4      1996-10-01  37.6771  128.7183           4  3.5
...           ...      ...       ...         ...  ...
571115 2009-12-31  35.5651  128.1699      571115  NaN
571116 2009-12-31  35.4915  128.7441      571116  NaN
571117 2009-12-31  35.4130  127.8791      571117  0.1
571118 2009-12-31  34.8882  128.6046      571118  NaN
571119 2009-12-31  34.8166  127.9264      571119  0.0

[571120 rows x 5 columns]


In [27]:
df_filled.groupby('time').size()
# df_filled.loc[(df_filled.time > '2000-01-01') & (df_filled.time < '2008-12-31')]

time
1996-10-01    118
1996-10-02    118
1996-10-03    118
1996-10-04    118
1996-10-05    118
             ... 
2009-12-27    118
2009-12-28    118
2009-12-29    118
2009-12-30    118
2009-12-31    118
Length: 4840, dtype: int64

In [1]:
import torch
import torch.nn.functional as F

def to_3ch_60x60(img_tensor: torch.Tensor)->torch.Tensor:
    x = img_tensor

    if x.ndim == 2:
        x = x.unsqueeze(0)
    elif x.ndim == 3:
        if x.shape[0] not in (1,3) and x.shape[2] in (1, 3):
            x = x.permute(2, 0, 1)
    else:
        raise ValueError(f'Unsupported ndim: {x.ndim}')
    
    x = x.float()

    C, H, W =x.shape
    if C == 1:
        x = x.repeat(3, 1, 1)
    elif C == 2:
        gray = x.mean(dim=0, keepdim=True)
        x = gray.repeat(3, 1, 1)
    elif C > 3:
        x = x[:3, ...]

    # 해상도 60x60으로 리사이즈
    x = F.interpolate(x.unsqueeze(0), size=(60,60),
                      mode='bilinear', align_corners=False).squeeze(0)
    return x

In [4]:
data = [0, 1.2, 3.4, 5.5, 13.0, 15.0, 0, 0, 3.4, 2.2, 0.345]

import numpy as np

data = np.array(data)
min_val = data.min()
max_val = data.max()

img_data = (data - min_val) / (max_val - min_val) * 255
img_tensor = torch.tensor(img_data, dtype=torch.float32)

# if img_tensor.numel() != 60*60:
#     raise ValueError('error')

img_tensor = img_tensor.view(60, 60)
img_tensor = img_tensor.unsqueeze(0).repeat(3, 1, 1)
print(img_tensor)

img_tensor = img_tensor/255

img_tensor = to_3ch_60x60(img_tensor)

RuntimeError: shape '[60, 60]' is invalid for input of size 11

# Point-wise data generation

In [ ]:
import os
from tqdm import tqdm
from PIL import Image
import numpy as np
import pandas as pd

device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')

# 설정
CSV_PATH = "data/preprocessed_obs_2010_2019.csv"
SAVE_DIR = "load/climate/obs_1xN_2010"

obs_data = pd.read_csv(CSV_PATH, encoding= 'cp949')
obs_station = obs_data.groupby("time").count()

H, W = 1, obs_station
LAT_MIN, LAT_MAX = 20, 79
LON_MIN, LON_MAX = 100, 159
NORM_MODE = "global"
FLIP_UD = True

vmin = float(val.min())
vmax = float(val.max()) + 1e-12
val01 = (val - vmin) / (vmax - vmin)
val255 = (val01 * 255.0).clip(0, 255).astype(np.uint8)

# 스트립 단일 이미지 생성
strip = np.tile(val255[None, :], (1,1))
strip_rgb = np.stack([strip, strip, strip], axis=-1)
img = Image.fromarray(strip_rgb, mode="RGB")
img.save('obs_strip.png')

def to_norm_xy(lon, lat, lon_min, lon_max, lat_min, lat_max):
    rx = (lon - lon_min) / (lon_max - lon_min)
    ry = (lat - lat_min) / (lat_max - lat_min)
    x = 2.0 * rx - 1.0
    y = 2.0 * ry - 1.0
    return x, y

x_norm, y_nornm = to_norm_xy(lon, lat, LON_MIN, LON_MAX, LAT_MIN, LAT_MAX)
 


os.makedirs(SAVE_DIR, exist_ok=True)

